# Calculate stage drift (DAPI/tissue-channel registration)

Re-images a small number of already-imaged FOVs to measure how far the stage
has drifted since an existing round was acquired, then applies the measured
correction to the experiment's **whole** positions file.

**Registers on `REGISTRATION_COLOR_NM` (default 405/DAPI) at a user-specified
`REGISTRATION_Z_UM`**, not a bead channel -- see the sibling
`stage_drift_beads.ipynb` for the bead-auto-detect version. Split into two
notebooks because they diverge in a real way, not just a parameter default:
bits rounds typically carry no DAPI/tissue channel at all, so the
bead-based notebook's "compare across cells + bits rounds" diagnostic doesn't
apply here (it previously crashed with `Could not auto-detect a
single-z-plane fiducial colour` the moment it reached a bits round with no
such channel) -- this notebook's own section 8 instead compares the exact
reference/new frame pair section 7 registered, which needs nothing from any
other round.

Two-part notebook, split by a manual step (running the generated Dave recipe on
the microscope):

**Part A -- before imaging.** Pick a reference round (an already-acquired data
folder: `cells`, `hybs/H01`, ...) -- `REFERENCE_FRAME_TABLE_PATH` can point at a
specific frame-table CSV instead of auto-resolving one from that round's
hal_config, for when the real frame table used at acquisition time is an
older/different one than metadata/ would auto-resolve to. Take FOVs
`FIRST_FOV .. FIRST_FOV+N_FOVS-1` (default 0..2) in the ORIGINAL experiment's
global fov_id numbering, and generate a tiny single-colour imaging round for
just those positions: a `positions_*_drift.txt` file, a HAL config + shutter
that image `REGISTRATION_COLOR_NM` + one blank frame at `REGISTRATION_Z_UM` --
nothing else -- and a minimal single-loop Dave recipe (no fluidics).
Everything is written to `SAMPLE_DIR/stage_drift/<same subpath as the
reference round's data_dir>/` (a folder at the same level as `MERci/`), e.g.
`stage_drift/cells/` or `stage_drift/hybs/H01/`.

**Dave numbers the resulting files locally, 0..N_FOVS-1, not by the original
fov_id.** Confirmed directly against storm_control's `v2Generator.py`
(`prompt_history/2026_07_08_1557_investigate_dave_fov_index_range.md`): every
loop's movie counter resets to 0 and zero-pads only as wide as that loop's own
position count -- never the original experiment's fov_id or its (much wider)
pad width. Section 5 builds the new series pattern/expected filenames from
`fov_pad_width(N_FOVS)` and local indices accordingly, not by reusing the
reference round's own (differently-sized) series pattern.

**-- run the generated Dave recipe now --** it steps through the `N_FOVS`
position(s) and writes the new registration images to `stage_drift/.../data/`.

**Part B -- after imaging.** For each of the `N_FOVS` FOVs, registers the new
frame against the reference round's own frame at the SAME colour/z for that FOV
via `skimage.registration.phase_cross_correlation` -- the same primitive
fishtank's own `align_experiments` coarse-alignment step uses (already
implemented in this repo's `MERci.acquisition.alignment.phase_drift`, built for
exactly this kind of registration; reused here rather than re-implemented),
after removing fixed hot/dead camera pixels that would otherwise dominate the
correlation and pin a dim channel's drift to exactly `[0, 0]`
(`acquisition.alignment.remove_hot_pixels`, section 7). The `N_FOVS` per-FOV
shifts are combined into **one** stage-wide translation `(tx, ty)` (median,
with the per-FOV spread reported as a QC check), which is then added to
**every** FOV in the experiment's positions file, written out as a new
drift-corrected positions file for use by the next real imaging round. Section
8 is a standalone diagnostic -- the exact reference/new frame pair, side by
side, for every FOV -- for when the numbers alone (zero or otherwise
suspicious drift) aren't enough to tell whether there's a real, visible
signal to register on, and whether it actually looks shifted between the two.

Scope: single-positions-file (legacy/single-tissue) layout only, matching
`measure_tissue_thickness_test.ipynb`'s own assumption -- the multi-boundary
per-segment `positions_file` layout is not handled here.


## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config       import ExperimentConfig
from MERci.common.metadata     import ExperimentMetadata
from MERci.common.io           import load_positions, save_positions_array, read_image_frames
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.acquisition.configs import (
    get_frame_table, get_color_sequence_name, hal_config_filename, shutter_filename,
    frame_table_filename, create_hal_config, create_shutter_file,
    find_frame_table_for_hal_config, read_hal_exposure_time, get_camera_pixel_size_um,
)
from MERci.acquisition.dave      import create_dave_config, series_to_movie_name, fov_pad_width
from MERci.acquisition.alignment import phase_drift, remove_hot_pixels
from MERci.transfer              import relative_to_data_root

print(f"SAMPLE_DIR : {SAMPLE_DIR}")


## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Used for the colour->hardware-channel map, camera pixel size (drift px->um
# conversion), and the Dave-recipe storage estimate.
MICROSCOPE = "ST2"

# Which already-imaged round to use as "the data" reference -- by imaging_type
# (e.g. "cells", "bits"), or set ROUND_ID directly to override. The next cell
# prints every round actually available before you pick.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Path to a specific frame-table CSV to use for the reference round, bypassing
# auto-resolution from its HAL config's <shutters> reference
# (find_frame_table_for_hal_config). Set this when the real frame table that was
# actually used for that round's acquisition is an older/different one than
# metadata/ would auto-resolve to (metadata/round_info can drift out of sync
# with what was really on the microscope at acquisition time). None (default)
# auto-resolves as before.
REFERENCE_FRAME_TABLE_PATH = None

# Re-image FOV ids FIRST_FOV .. FIRST_FOV+N_FOVS-1 (in the original experiment's
# global fov_id numbering) as the drift check.
FIRST_FOV = 0
N_FOVS    = 3

# Registration colour (nm) for the drift-check round. Default 405 (DAPI/tissue),
# not the reference round's own bead colour (e.g. 488): a dim bead channel can
# still be unreliable to register on even after remove_hot_pixels (section 7),
# whereas DAPI/tissue signal is often much stronger. See the sibling
# stage_drift_beads.ipynb for the bead-based approach instead.
REGISTRATION_COLOR_NM = 405.0

# z (um, same convention as the reference round's own frame table) to image the
# drift-check round at, and to match against in the reference round's own
# z-stack -- pick a z where REGISTRATION_COLOR_NM actually shows real signal
# (e.g. from section 8's frame comparison, or
# measure_tissue_thickness_test.ipynb's z_first_um/z_last_um for this FOV/round).
# No sensible default exists across experiments -- this MUST be set explicitly.
REGISTRATION_Z_UM = None

# Laser power for the drift-check registration frame (arbitrary units, same
# convention as acquisition.configs.create_shutter_file's default_power). Not
# read back from the reference round's own shutter XML (no such reader exists)
# -- review the resulting image brightness once acquired and adjust if needed.
REGISTRATION_POWER = 1.0

# Subpixel registration precision (1/upsample_factor px); 10 -> 0.1 px, the
# same default acquisition.alignment.compute_fov_drifts uses.
UPSAMPLE_FACTOR = 10

# +1.0 (default) assumes image (row, col) axes map onto stage (y, x) axes with
# no flip when converting the measured pixel shift to a stage-um correction
# (section 7/9) -- the same unverified default acquisition.alignment.
# compute_fov_drifts's own sign_x/sign_y flag explicitly, since the true
# mapping is specific to the microscope's camera<->stage convention. Confirm
# against a FOV with an obvious real drift before trusting section 10; set to
# -1.0 if the correction turns out backwards.
DRIFT_SIGN = 1.0

# Flag per-FOV disagreement (section 9) once the max-min spread across FOVs
# exceeds this many pixels' worth of stage motion -- a single rigid whole-round
# shift should register every FOV to within a fraction of a pixel of the same
# (tx, ty); a larger spread means something other than uniform stage drift
# (e.g. a bad bead detection on one FOV) and the combined correction below
# should not be trusted without checking fov_drift_measurements.csv first.
SPREAD_WARNING_PX = 2.0

# Explicit plot font sizes (NOTEBOOK_GUIDELINES.md #5).
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 11
PLOT_LEGEND_FONTSIZE = 10

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Positions tag       : {POSITIONS_TAG}")
print(f"Microscope         : {MICROSCOPE}")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"FOVs               : {FIRST_FOV} .. {FIRST_FOV + N_FOVS - 1}  (N_FOVS={N_FOVS})")


## 3 — Pick the reference round ("get a data folder")

In [ ]:
config = ExperimentConfig(
    data_dir       = SAMPLE_DIR / "data",
    metadata_dir   = SAMPLE_DIR / "metadata",
    analysis_dir   = SAMPLE_DIR / "analysis",
    settings_dir   = SAMPLE_DIR / "settings",
    round_info_csv = SAMPLE_DIR / "metadata" / "round_info.csv",
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                image_suffix=config.image_suffix)

print("Available rounds (data folders):")
for rid in meta.valid_round_ids():
    series_list = meta.series_for_round(rid)
    types = sorted({(s.imaging_type or "?") for s in series_list})
    dirs  = sorted({str(s.data_dir) for s in series_list if s.data_dir is not None})
    print(f"  round {rid:>2}  imaging_type={types}  dir={dirs or ['(SAMPLE_DIR/data)']}")


def resolve_round_id(meta, imaging_type):
    """First round_id whose series carry the given imaging_type."""
    for rid in meta.valid_round_ids():
        if any((s.imaging_type or "").strip().lower() == imaging_type.strip().lower()
               for s in meta.series_for_round(rid)):
            return rid
    raise ValueError(f"No round found with imaging_type={imaging_type!r}")


def load_round_frame_table(round_id, config, meta, frame_table_path=None):
    """(frame_table, hal_config_path, SeriesInfo) for round_id's first series with a
    hal_config. If frame_table_path is given, read that CSV directly instead of
    auto-resolving one from the HAL config's <shutters> reference -- for when the
    frame table actually used for that round's real acquisition is an older/
    different one than metadata/ would auto-resolve to."""
    for s in meta.series_for_round(round_id):
        if not s.hal_config:
            continue
        hal_path = Path(config.settings_dir) / s.hal_config
        if frame_table_path is not None:
            return pd.read_csv(frame_table_path, index_col=0), hal_path, s
        ft_path = find_frame_table_for_hal_config(hal_path, config.metadata_dir)
        if ft_path and ft_path.exists():
            return pd.read_csv(ft_path, index_col=0), hal_path, s
    raise FileNotFoundError(f"No frame table found for round {round_id}")


def nearest_frame_for_color_z(frame_table, color_nm, z_um):
    """Frame index of the given colour's frame with z closest to z_um."""
    candidates = frame_table[frame_table["color"].round(0) == round(color_nm)]
    if candidates.empty:
        raise ValueError(f"No frames of colour {color_nm:.0f} nm in this frame table.")
    return int((candidates["z"] - z_um).abs().idxmin())


target_round_id = ROUND_ID if ROUND_ID is not None else resolve_round_id(meta, ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"the FOVs selected below may not all have real data yet.")

if REGISTRATION_Z_UM is None:
    raise ValueError(
        "REGISTRATION_Z_UM is not set (section 2) -- pick a z (um) where "
        f"REGISTRATION_COLOR_NM ({REGISTRATION_COLOR_NM:.0f} nm) actually shows real "
        "signal (e.g. from section 8's frame comparison, or "
        "measure_tissue_thickness_test.ipynb's z_first_um/z_last_um) before continuing."
    )

frame_table, reference_hal_config_path, reference_series = load_round_frame_table(
    target_round_id, config, meta, frame_table_path=REFERENCE_FRAME_TABLE_PATH,
)
reference_data_dir      = reference_series.data_dir or config.data_dir
reference_exposure_time = read_hal_exposure_time(reference_hal_config_path) or 0.25

reference_frame_idx = nearest_frame_for_color_z(frame_table, REGISTRATION_COLOR_NM, REGISTRATION_Z_UM)
reference_frame_z   = float(frame_table.loc[reference_frame_idx, "z"])

print(f"\nTarget round         : {target_round_id}  (series pattern: {reference_series.name!r})")
print(f"Reference hal_config : {reference_hal_config_path}")
print(f"Reference frame table: {REFERENCE_FRAME_TABLE_PATH or '(auto-resolved from hal_config)'}")
print(f"Reference data_dir   : {reference_data_dir}")
print(f"Reference frame      : index {reference_frame_idx}, colour {REGISTRATION_COLOR_NM:.0f} nm, "
      f"z={reference_frame_z:.2f} um (requested z={REGISTRATION_Z_UM:.2f} um)")


## 4 — Select FOVs FIRST_FOV..FIRST_FOV+N-1 and build the drift-check round

Generates the new positions file, frame table (`REGISTRATION_COLOR_NM` + one
blank, at `REGISTRATION_Z_UM`), HAL config, and shutter file for the drift
check -- written to `SAMPLE_DIR/stage_drift/<reference round's own data subpath>/`
(e.g. `stage_drift/cells/`), a folder at the same level as `MERci/`.


In [ ]:
# ---- FOV selection + subset positions file --------------------------------
FOV_IDS = list(range(FIRST_FOV, FIRST_FOV + N_FOVS))

full_positions = load_positions(config.positions_txt)   # {fov_id: (x, y)}
missing_fovs = [f for f in FOV_IDS if f not in full_positions]
if missing_fovs:
    raise ValueError(f"FOV id(s) {missing_fovs} not found in {config.positions_txt}")

drift_positions_coords = np.array([full_positions[f] for f in FOV_IDS], dtype=float)

# ---- Drift-check frame table: registration colour + one blank, at REGISTRATION_Z_UM ----
drift_frame_table = get_frame_table(
    bead_z=REGISTRATION_Z_UM, bead_seq=[REGISTRATION_COLOR_NM, np.nan], color_seq=[], end_seq=[],
    z_pos=np.array([]), microscope=MICROSCOPE, z_return_mode="instant",
)
drift_name = get_color_sequence_name(drift_frame_table)
print(f"Drift-check frame table ({len(drift_frame_table)} frame(s)):")
print(drift_frame_table)
print(f"Sequence name: {drift_name!r}")

# ---- Output folder: stage_drift/<reference round's own data_dir subpath> --
subpath = relative_to_data_root(Path(reference_data_dir))
if subpath.parts and subpath.parts[0].lower() == "data":
    subpath = subpath.relative_to("data")
stage_drift_dir      = SAMPLE_DIR / "stage_drift" / subpath
stage_drift_data_dir = stage_drift_dir / "data"
stage_drift_dir.mkdir(parents=True, exist_ok=True)
stage_drift_data_dir.mkdir(parents=True, exist_ok=True)
print(f"\nStage-drift folder: {stage_drift_dir}")

# ---- Write positions / hal_config / shutter / frame table -----------------
drift_positions_path = stage_drift_dir / f"positions_{POSITIONS_TAG}_drift.txt"
save_positions_array(drift_positions_coords, drift_positions_path)

drift_hal_config_name  = hal_config_filename(MICROSCOPE, "drift", drift_name)
drift_shutter_name      = shutter_filename("drift", drift_name)
drift_frame_table_name  = frame_table_filename("drift", drift_name)

drift_hal_config_path   = stage_drift_dir / drift_hal_config_name
drift_shutter_path      = stage_drift_dir / drift_shutter_name
drift_frame_table_path  = stage_drift_dir / drift_frame_table_name

create_shutter_file(drift_frame_table, drift_shutter_path, default_power=REGISTRATION_POWER)
create_hal_config(
    reference_hal_config_path, drift_frame_table, drift_shutter_name, drift_hal_config_path,
    file_type=IMAGE_SUFFIX, exposure_time=reference_exposure_time,
)
drift_frame_table.to_csv(drift_frame_table_path)

print(f"Wrote positions file : {drift_positions_path}  ({len(FOV_IDS)} FOV(s): {FOV_IDS})")
print(f"Wrote HAL config     : {drift_hal_config_path}")
print(f"Wrote shutter file   : {drift_shutter_path}")
print(f"Wrote frame table    : {drift_frame_table_path}")


## 5 — Build the Dave recipe (single loop, no fluidics)

In [ ]:
# Dave numbers movies with a LOCAL, 0-based-per-loop counter, zero-padded to
# just wide enough for THIS loop's own position count -- never the original
# experiment's global fov_id or its (much wider) pad width. Confirmed directly
# against storm_control's v2Generator.py: `copyChildren` appends
# `_ + str(loop_iterator).zfill(pad_length)`, where `loop_iterator` resets to 0
# for every loop and `pad_length = len(str(num_positions_in_that_loop))` (see
# prompt_history/2026_07_08_1557_investigate_dave_fov_index_range.md). Reusing
# the reference round's own series pattern (e.g. "..._{fov:03d}", sized for
# its ~1000+ FOVs) would therefore predict filenames Dave never actually
# writes for this tiny N_FOVS-position loop -- fov_pad_width(N_FOVS) matches
# Dave's real per-loop width instead.
drift_pad             = fov_pad_width(N_FOVS)
drift_series_pattern  = f"{series_to_movie_name(reference_series.name)}_{{fov:0{drift_pad}d}}"

drift_round_info = pd.DataFrame([{
    "imaging_round": 1,
    "imaging_type":  "drift",
    "series":        drift_series_pattern,
    "hal_config":    drift_hal_config_name,
    "data_dir":      str(stage_drift_data_dir),
    # Explicit, not relied-upon-by-default: pins the local counter to start at
    # 0 and pad to drift_pad via the patched Dave's <name start=".." pad="..">
    # attributes (misc/dave_multi_z/README.md) if deployed; harmless/ignored
    # by a stock Dave, whose own default numbering already matches these
    # values for a non-power-of-10 FOV count like this.
    "fov_start":     0,
    "fov_pad":       drift_pad,
}])

dave_output_path = stage_drift_dir / f"dave-{MICROSCOPE.lower()}-drift-{SAMPLE_NAME}.xml"

create_dave_config(
    drift_round_info,
    positions_file   = drift_positions_path,
    settings_dir     = stage_drift_dir,
    output_path      = dave_output_path,
    kilroy_config    = None,     # no fluidics in a single-colour round
    create_data_dirs = True,
    print_estimate   = False,
    microscope       = MICROSCOPE,
)

# LOCAL index (0..N_FOVS-1, matching Dave's own per-loop counter) -- NOT the
# original global fov_id -- maps 1:1 to FOV_IDS in positions-file line order.
new_paths = {
    fov_id: stage_drift_data_dir / f"{drift_series_pattern.format(fov=local_idx)}{IMAGE_SUFFIX}"
    for local_idx, fov_id in enumerate(FOV_IDS)
}

print(f"Wrote Dave recipe: {dave_output_path}")
print(f"\n--- NEXT STEP (manual) ---")
print(f"Load {dave_output_path.name} in Dave and run it. It will step through the {len(FOV_IDS)} "
      f"position(s) in {drift_positions_path.name} and write {REGISTRATION_COLOR_NM:.0f} nm "
      f"images (at z={REGISTRATION_Z_UM:.2f} um) to:")
print(f"  {stage_drift_data_dir}")
print(f"Expected file(s) (locally numbered 0..{len(FOV_IDS) - 1}, NOT the original fov_id "
      f"{FOV_IDS[0]}..{FOV_IDS[-1]}):")
for fov_id in FOV_IDS:
    print(f"  {new_paths[fov_id]}")


---
## STOP -- run the Dave recipe above on the microscope now

Once the `N_FOVS` bead image(s) listed above exist on disk, continue with Part B
below.
---


## 6 — Part B: locate the reference and new bead images

In [ ]:
ref_paths = {fov_id: reference_series.resolve_path(fov_id, config.image_suffix) for fov_id in FOV_IDS}

missing_ref = [f for f, p in ref_paths.items() if not p.exists()]
missing_new = [f for f, p in new_paths.items() if not p.exists()]
if missing_ref:
    raise FileNotFoundError(f"Reference image(s) missing for FOV(s) {missing_ref}: "
                             f"{[str(ref_paths[f]) for f in missing_ref]}")
if missing_new:
    raise FileNotFoundError(
        f"New drift-check image(s) missing for FOV(s) {missing_new} -- "
        f"run the Dave recipe from section 5 first.\n"
        f"Expected: {[str(new_paths[f]) for f in missing_new]}"
    )

print(f"Found {len(FOV_IDS)} reference + {len(FOV_IDS)} new registration image(s).")


## 7 — Per-FOV drift (phase cross-correlation)

Registers each FOV's new registration frame onto its reference frame with
`skimage.registration.phase_cross_correlation` (`acquisition.alignment.
phase_drift`) -- the same primitive fishtank's `align_experiments` script uses
for its own coarse bead-based registration; MERlin has no equivalent, so this
reuses the already-implemented fishtank-style approach rather than adding a new
one. Registering on `REGISTRATION_COLOR_NM` (default 405/DAPI) at
`REGISTRATION_Z_UM` rather than the bead channel avoids relying on bead
signal that can be too dim to register on reliably even after the fix below.

**Hot/dead-pixel removal before registration.** A fixed detector defect (a hot
or dead camera pixel) sits at the same `(row, col)` in every frame, so when the
real signal is dim, that single fixed pixel becomes the brightest,
most-repeated feature and dominates `phase_cross_correlation`, pinning the
recovered shift to exactly `[0, 0]` -- the exact failure this repo's own
fishtank fork (`fix_reg_hot_pixels.patch`/`pr_body_zero_drift.md`) diagnosed and
fixed. `acquisition.alignment.remove_hot_pixels` (ported from that fix) is
applied to both the reference and new frame below before registration: it
replaces isolated pixels far above their own local median with that median,
leaving real (multi-pixel) tissue/bead signal untouched. It only catches
ISOLATED pixel-scale spikes, though -- a broader fixed pattern (vignetting, a
dust artifact on the optics) can dominate the same way and would not be
touched by it.

Still getting zero/suspicious translation after this? Section 8 below shows
the exact reference/new frame pair, side by side, for every FOV -- use it to
check whether there's a real, visible signal at all, and whether it visibly
sits in a different place between the two frames, before assuming the
registration math is wrong.

**Sign convention -- confirm against real data before trusting section 10.**
`phase_drift` returns the pixel shift that, applied to the *new* frame,
registers it onto the *reference* frame; converting that directly to stage
micrometers (as done below) assumes image (row, col) axes map onto stage
(y, x) axes with **no flip** -- the same unverified default
`acquisition.alignment.compute_fov_drifts`'s own `sign_x`/`sign_y` parameters
flag explicitly, since the true mapping depends on the specific microscope's
camera<->stage convention. Confirm the sign is right on a FOV with an obvious,
large real drift (e.g. by eye, comparing the reference and new images) before
applying section 10's whole-positions-file correction; flip the sign of
`TX_UM`/`TY_UM` below if it turns out backwards.


In [ ]:
PIXEL_SIZE_UM             = get_camera_pixel_size_um(MICROSCOPE)
NEW_REGISTRATION_FRAME_IDX = 0   # drift_frame_table's frame 0 is REGISTRATION_COLOR_NM (section 4)

drift_rows = []
n_zero_shift = 0
for fov_id in FOV_IDS:
    ref_frame = read_image_frames(ref_paths[fov_id], [reference_frame_idx],
                                   frame_width=config.frame_width, frame_height=config.frame_height)[0]
    new_frame = read_image_frames(new_paths[fov_id], [NEW_REGISTRATION_FRAME_IDX],
                                   frame_width=config.frame_width, frame_height=config.frame_height)[0]
    shift, error = phase_drift(
        remove_hot_pixels(ref_frame), remove_hot_pixels(new_frame), upsample_factor=UPSAMPLE_FACTOR,
    )
    dy_px, dx_px = float(shift[0]), float(shift[1])
    if dy_px == 0.0 and dx_px == 0.0:
        n_zero_shift += 1
    drift_rows.append({
        "fov_id":     fov_id,
        "dy_px":      dy_px,
        "dx_px":      dx_px,
        "error":      error,
        "drift_x_um": dx_px * PIXEL_SIZE_UM * DRIFT_SIGN,
        "drift_y_um": dy_px * PIXEL_SIZE_UM * DRIFT_SIGN,
    })

drift_df = pd.DataFrame(drift_rows)
drift_csv = stage_drift_dir / "fov_drift_measurements.csv"
drift_df.to_csv(drift_csv, index=False)
print(f"Saved per-FOV drift measurements: {drift_csv}")

if n_zero_shift:
    # An EXACT [0, 0] shift (not just small) is the tell-tale sign of a still-
    # dominant fixed artifact rather than genuinely zero drift -- remove_hot_pixels
    # above handles the known hot/dead-pixel case, but a saturated real signal, a
    # bad flat-field, or some other fixed pattern could still produce this.
    print(f"\nWARNING: {n_zero_shift}/{len(drift_df)} FOV(s) registered at EXACTLY "
          f"(dy_px, dx_px) = (0.0, 0.0) -- almost always spurious (a fixed image "
          f"feature dominating the correlation), not genuinely zero drift. Inspect "
          f"those FOVs' reference/new frames directly before trusting them.")


## 8 — Diagnostic: reference vs. new frame, side by side

Still seeing zero (or suspicious) translation after section 7? Look at the
exact two frames `phase_drift` compared, side by side, for every FOV -- not a
cross-round bead comparison (bits rounds typically carry no
`REGISTRATION_COLOR_NM` channel at all, so that comparison doesn't apply here;
see the sibling `stage_drift_beads.ipynb` for that version). This answers what
the numbers alone can't: is there real, visible signal in *both* frames, does
it look like it's actually in a different place between them (contradicting an
exact-zero result), and does `remove_hot_pixels` leave anything suspicious
behind (e.g. a broad fixed pattern -- vignetting, a dust artifact -- that isn't
an isolated pixel spike and so isn't touched by it)?


In [ ]:
# ---- Calculation --------------------------------------------------------
compare_frames = {}   # fov_id -> {"ref": raw, "new": raw} (the exact frames section 7 registered)
for fov_id in FOV_IDS:
    compare_frames[fov_id] = {
        "ref": read_image_frames(ref_paths[fov_id], [reference_frame_idx],
                                  frame_width=config.frame_width, frame_height=config.frame_height)[0],
        "new": read_image_frames(new_paths[fov_id], [NEW_REGISTRATION_FRAME_IDX],
                                  frame_width=config.frame_width, frame_height=config.frame_height)[0],
    }
print(f"Read reference + new frame for {len(compare_frames)} FOV(s).")


In [ ]:
# ---- Display --------------------------------------------------------------
n_rows = len(FOV_IDS)
fig, axes = plt.subplots(n_rows, 2, figsize=(8, 4 * n_rows), squeeze=False)

drift_by_fov = drift_df.set_index("fov_id")   # from section 7, always populated by now

for row, fov_id in enumerate(FOV_IDS):
    for col, key in enumerate(("ref", "new")):
        ax = axes[row][col]
        frame = compare_frames[fov_id][key]
        n_hot = int(np.sum(remove_hot_pixels(frame) != frame))
        lo, hi = np.percentile(frame, [1, 99.5])
        ax.imshow(frame, cmap="gray", vmin=lo, vmax=max(hi, lo + 1))
        title = "reference" if key == "ref" else "new"
        ax.set_title(
            f"FOV {fov_id} -- {title}\n"
            f"min={int(frame.min())} mean={frame.mean():.0f} max={int(frame.max())} hot_px={n_hot}",
            fontsize=PLOT_TICK_FONTSIZE,
        )
        ax.axis("off")
    row_data = drift_by_fov.loc[fov_id]
    axes[row][0].set_ylabel(
        f"dy={row_data['dy_px']:.2f}px dx={row_data['dx_px']:.2f}px", fontsize=PLOT_TICK_FONTSIZE,
    )

fig.suptitle("Reference vs. new frame per FOV (per-tile percentile stretch)", fontsize=PLOT_TITLE_FONTSIZE)
fig.tight_layout()

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
compare_fig_path = figures_dir / "stage_drift_ref_vs_new_comparison.png"
fig.savefig(compare_fig_path, dpi=150)
plt.show()
print(f"Saved: {compare_fig_path}")


## 9 — Combine into one stage-wide translation

In [ ]:
print(drift_df.to_string(index=False))

TX_UM = float(drift_df["drift_x_um"].median())
TY_UM = float(drift_df["drift_y_um"].median())
spread_x_um = float(drift_df["drift_x_um"].max() - drift_df["drift_x_um"].min())
spread_y_um = float(drift_df["drift_y_um"].max() - drift_df["drift_y_um"].min())

print(f"\nCombined translation (median across {len(drift_df)} FOV(s)): "
      f"tx={TX_UM:.3f} um, ty={TY_UM:.3f} um")
print(f"Spread across FOVs (max-min): dx={spread_x_um:.3f} um, dy={spread_y_um:.3f} um")

spread_warning_um = SPREAD_WARNING_PX * PIXEL_SIZE_UM
if max(spread_x_um, spread_y_um) > spread_warning_um:
    print(f"\nWARNING: the {len(drift_df)} FOV(s) do not agree on a single translation "
          f"(spread > {SPREAD_WARNING_PX:.1f} pixel(s) = {spread_warning_um:.3f} um) -- this may not "
          f"be a simple whole-round shift. Inspect {drift_csv.name} before trusting the correction below.")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(drift_df["drift_x_um"], drift_df["drift_y_um"], s=60, color="tab:blue",
           label="per-FOV measurement", zorder=3)
for _, row in drift_df.iterrows():
    ax.annotate(f"FOV {int(row['fov_id'])}", (row["drift_x_um"], row["drift_y_um"]),
                textcoords="offset points", xytext=(6, 6), fontsize=PLOT_TICK_FONTSIZE)
ax.scatter([TX_UM], [TY_UM], s=140, color="tab:red", marker="x", label="combined (median)", zorder=4)
ax.axhline(0, color="gray", lw=0.5)
ax.axvline(0, color="gray", lw=0.5)
ax.set_xlabel("drift_x (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("drift_y (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Per-FOV bead drift, round {target_round_id}", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
ax.set_aspect("equal", adjustable="datalim")
fig.tight_layout()

figures_dir = config.analysis_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
fig_path = figures_dir / f"stage_drift_round{target_round_id}.png"
fig.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved: {fig_path}")


## 10 — Apply the translation to the whole positions file

Adds `(tx, ty)` to **every** FOV in `positions_{POSITIONS_TAG}.txt` (not just the
`N_FOVS` sampled above) and writes the result as a new, drift-corrected
positions file for use by the next real imaging round.

In [ ]:
corrected = {fov_id: (x + TX_UM, y + TY_UM) for fov_id, (x, y) in full_positions.items()}
corrected_coords = np.array([corrected[fov_id] for fov_id in sorted(corrected)], dtype=float)

corrected_path = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}_drift_corrected_round{target_round_id}.txt"
save_positions_array(corrected_coords, corrected_path)

print(f"Applied translation (tx={TX_UM:.3f} um, ty={TY_UM:.3f} um) to all {len(full_positions)} "
      f"FOV(s) from {config.positions_txt.name}.")
print(f"Wrote drift-corrected positions file: {corrected_path}")
print("Use this file as the positions file for the NEXT Dave round to compensate for the measured stage drift.")
